In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 107 (delta 32), reused 91 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 19.81 MiB | 19.96 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [2]:
import os

# ЭМУЛЯЦИЯ 8 ЯДЕР CPU
# Это нужно сделать ДО импорта JAX!
#os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import sys

# Проверка
print(f"JAX Version: {jax.__version__}")
print(f"Devices Available: {jax.devices()}") # Должно показать 8 cpu
print(f"Platform: {jax.devices()[0].platform.upper()}")

JAX Version: 0.7.2
Devices Available: [CudaDevice(id=0)]
Platform: GPU


In [3]:
try:
    from core import Pars1d, Equation1d
    from equations import make_burgers_1d
    from solver import Solver1d
    from parallel_solver import ParallelSolver1d
    from benchmark_1d import run_solver_benchmark
    print("Library imported successfully!")
except ImportError as e:
    print(f"Error importing library: {e}")
    print("Make sure you uploaded core.py, solver.py, etc. to the Colab files section!")

Library imported successfully!


In [4]:
from typing import Tuple
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

class FastSolver1d(Solver1d):
    """
    Оптимизированная версия солвера для бенчмарков.
    Использует jax.lax.while_loop для выполнения всего цикла на устройстве.
    Не сохраняет промежуточные шаги (только начальное и конечное состояние).
    """
    def __init__(self, pars, eqn, scheme_name="sd2"):
        super().__init__(pars, eqn, scheme_name)

        self.solve_jit = jax.jit(self._solve_internal)

    def _solve_internal(self, u0):

        def cond_fun(state):
            t, _, _ = state
            return t < self.pars.t_final

        def body_fun(state):
            t, u, step_idx = state

            max_speed = jnp.max(self.eqn.spectral_radius(u))
            safe_speed = jnp.maximum(max_speed, 1e-6)
            dt = self.pars.cfl * self.pars.dx / safe_speed

            dt = jnp.minimum(dt, self.pars.t_final - t)

            u_new = self.step_fn(t, u, dt, self.rhs_fn)

            return t + dt, u_new, step_idx + 1

        init_state = (0.0, u0, 0)
        final_t, final_u, final_steps = jax.lax.while_loop(cond_fun, body_fun, init_state)

        return final_t, final_u, final_steps

    def solve(self):
        dx = self.pars.dx
        x = jnp.linspace(self.pars.x_init + dx/2, self.pars.x_final - dx/2, self.pars.J)
        u0 = self.eqn.initial_data(x)
        
        final_t, final_u, steps = self.solve_jit(u0)


        return {
            "x": x,
            "t": jnp.array([0.0, final_t]),
            "u_n": jnp.stack([u0, final_u]),
            "steps": steps
        }


In [12]:
# --- CPU benchmarks ---
print("=== GPU BENCHMARK START ===")

#GRIDS = [1000, 2000, 4000, 8000]
GRIDS = [1000, 5000, 10000]

eqn = make_burgers_1d()

df_jax_gpu_fast = run_solver_benchmark(FastSolver1d, eqn, "sd2", "JAX (GPU)", GRIDS)
df_jax_gpu_standart = run_solver_benchmark(Solver1d, eqn, "sd2", "JAX (GPU)", GRIDS)

#df_final = pd.concat([df_centpy, df_jax_single, df_jax_parallel])
df_final = pd.concat([df_jax_gpu_fast, df_jax_gpu_standart])
filename = f"cpu_sd2_1d_burgers_{min(GRIDS)}_{max(GRIDS)}.csv"
df_final.to_csv(filename, index=False)

print(f"\n Benchmark Finished! Saved to {filename}")
print(df_final)


=== GPU BENCHMARK START ===

--- Benchmarking: JAX (GPU) ---
  Grid 1000: 0.0372 s (Warmup: 0.35 s)
  Grid 5000: 0.1585 s (Warmup: 0.56 s)
  Grid 10000: 0.2780 s (Warmup: 0.58 s)

--- Benchmarking: JAX (GPU) ---
Starting simulation: Burgers 1D
Grid: 1000 points, Scheme: SD2/minmod
Simulation finished in 0.5541s
Total steps: 484
Starting simulation: Burgers 1D
Grid: 1000 points, Scheme: SD2/minmod
Simulation finished in 0.2361s
Total steps: 484
  Grid 1000: 0.2384 s (Warmup: 0.56 s)
Starting simulation: Burgers 1D
Grid: 5000 points, Scheme: SD2/minmod
Simulation finished in 1.4762s
Total steps: 2421
Starting simulation: Burgers 1D
Grid: 5000 points, Scheme: SD2/minmod
Simulation finished in 1.0821s
Total steps: 2421
  Grid 5000: 1.0841 s (Warmup: 1.48 s)
Starting simulation: Burgers 1D
Grid: 10000 points, Scheme: SD2/minmod
Simulation finished in 2.5742s
Total steps: 4841
Starting simulation: Burgers 1D
Grid: 10000 points, Scheme: SD2/minmod
Simulation finished in 3.1012s
Total steps: 4